In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import h5py
import os
import seisbench.data as sbd
import matplotlib.pyplot as plt
import random
from pathlib import Path
from scipy import signal
from sklearn.preprocessing import StandardScaler
from scipy.fft import fft, ifft
from scipy.interpolate import CubicSpline
from scipy.signal import hilbert
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler 
from sklearn.metrics import accuracy_score, classification_report
import warnings
import time
warnings.filterwarnings('ignore')

# Step 1: Data prepare for SVM and RF
### The well prepared data of vcseis is provided with this notebook.
### The path of the data should be 
Training csv file: data/svmrf/train.csv  
Testing csv file: data/svmrf/test.csv  
H5 file: data/svmrf/total.hdf5  

In [2]:
datapath = Path('D:/seisbench_data/vcseis/')
exclude = {'npzdata', 'all_meta'}
folders = [item for item in datapath.iterdir() 
           if item.is_dir() and item.name not in exclude]
print(folders)


[WindowsPath('D:/seisbench_data/vcseis/ak_lp'), WindowsPath('D:/seisbench_data/vcseis/ak_ns'), WindowsPath('D:/seisbench_data/vcseis/ak_vt'), WindowsPath('D:/seisbench_data/vcseis/cas_lp'), WindowsPath('D:/seisbench_data/vcseis/cas_vt'), WindowsPath('D:/seisbench_data/vcseis/hw_lp'), WindowsPath('D:/seisbench_data/vcseis/hw_ns'), WindowsPath('D:/seisbench_data/vcseis/hw_vt'), WindowsPath('D:/seisbench_data/vcseis/nc_lp'), WindowsPath('D:/seisbench_data/vcseis/nc_vt')]


In [4]:
# data checking
cur_data = sbd.WaveformDataset(folders[1])
print(cur_data.metadata.columns.tolist())
if cur_data.get_waveforms(0)[1][0] == 0:
    print('y')

print(cur_data.get_waveforms(0).shape)

2026-06-19 22:04:12,676 | seisbench | WARNING | Output component order not specified, defaulting to 'ZNE'.


['index', 'source_id', 'source_origin_time', 'source_latitude_deg', 'source_longitude_deg', 'source_depth_km', 'source_magnitude', 'source_magnitude_type', 'source_type', 'station_network_code', 'station_code', 'station_location_code', 'trace_channel', 'station_latitude_deg', 'station_longitude_deg', 'station_elevation_m', 'station_epicentral_distance_m', 'path_azimuth_deg', 'path_back_azimuth_deg', 'trace_p_arrival_time', 'trace_s_arrival_time', 'trace_p_max_weight', 'trace_s_max_weight', 'trace_p_first_motion', 'trace_name', 'trace_sampling_rate_hz', 'trace_has_spikes', 'trace_start_time', 'trace_p_arrival_sample', 'trace_p_status', 'trace_s_arrival_sample', 'trace_s_status', 'trace_snr_db', 'trace_mean_snr_db', 'trace_frequency_index', 'split', 'trace_name_original', 'source_frequency_index', 'trace_chunk', 'trace_component_order']
y
(3, 11000)


# Data preprocess
In this step, we regenerate the metadata. Only the necessary information is kept. Besides, we also transpose the waveform data

In [ ]:
converted_unified_csv_frame_test = pd.DataFrame(columns=['Key', 'Type'])
converted_unified_csv_frame_train = pd.DataFrame(columns=['Key', 'Type'])

# train csv save path
LP_VT_train_csv_file_path = 'D:/seisbench_data/training.csv'
# load if csv file exists, otherwise generate it
if os.path.exists(LP_VT_train_csv_file_path):
    LPVT_train_csv = pd.read_csv(LP_VT_train_csv_file_path)
else:
    LPVT_train_csv = None
    pass

# test csv save path
LP_VT_test_csv_file_path = 'D:/seisbench_data/test.csv'
# load if csv file exists, otherwise generate it
if os.path.exists(LP_VT_test_csv_file_path):
    LPVT_test_csv = pd.read_csv(LP_VT_test_csv_file_path)
else:
    LPVT_test_csv = None
    pass

In [ ]:
LP_VT_hdf5_file_path = 'D:/seisbench_data/total.hdf5'
# append hdf5 file if it exists, otherwise create it
if os.path.exists(LP_VT_hdf5_file_path):
    LPVT_hdf5 = h5py.File(LP_VT_hdf5_file_path, 'a')
else:
    LPVT_hdf5 = h5py.File(LP_VT_hdf5_file_path, 'w')

In [ ]:
def csv_dealer(path, etype):
    cur_data = sbd.WaveformDataset(path)
    
    for instance_id in range(len(cur_data)):
        # exclude single z data
        if cur_data.get_waveforms(instance_id)[1][0] == 0 and cur_data.get_waveforms(instance_id)[2][0] == 0:
            continue
        
        source_id = cur_data['source_id'][instance_id]

        if instance_id % 2500 == 0:
            print('{} / {}'.format(instance_id, len(cur_data)))
        # all data here is earthquake data
        t_Type = etype


        # Eq_id
        t_Eq_id = t_Type + '.' + str(cur_data['source_id'].iloc[instance_id])
        
        # get data from downloaded file 
        t_data = cur_data.get_waveforms(instance_id).T
        print(len(t_data))
        print(len(t_data[0]))
        print(type(t_data))

        # Key is combination of From.Eq_id.Net_code.Sta_code.Instrument
        t_Key = t_Eq_id + '.' + str(instance_id)
        try:
            # append to LSD_hdf5, save as int32 to save space
            LPVT_hdf5.create_dataset(t_Key, data=t_data.astype(np.int32), compression='gzip', compression_opts=9)
        except:
            print('Key {} already exists in lp_vt.hdf5'.format(t_Key))
            # continue
        # append to csv_frame/ use concate to avoid warning
        # csv_frame = csv_frame.append({'Key': t_Key, 'Type': t_Type, 'P_index': t_P_index, 'Pn_index': t_Pn_index, 'Pg_index': t_Pg_index, 'S_index': t_S_index, 'Sn_index': t_Sn_index, 'Sg_index': t_Sg_index, 'P_polarity': t_P_polarity, 'Baz': t_Baz, 'Dis': t_Dis, 'Mag_value': t_Mag_value, 'Mag_type': t_Mag_type, 'Eq_depth': t_Eq_depth, 'Eq_lat': t_Eq_lat, 'Eq_lon': t_Eq_lon, 'Net_code': t_Net_code, 'Sta_code': t_Sta_code, 'Sta_lat': t_Sta_lat, 'Sta_lon': t_Sta_lon, 'Sta_ele': t_Sta_ele, 'Eq_time': t_Eq_time, 'Eq_id': t_Eq_id, 'From': t_From, 'Length': t_Length, 'Channel': t_Channel, 'Instrument': t_Instrument}, ignore_index=True)
        rand = random.randint(1, 10)
        if rand <= 7:
            global converted_unified_csv_frame_train
            converted_unified_csv_frame_train = pd.concat([converted_unified_csv_frame_train, pd.DataFrame({'Key': [t_Key], 'Type': [t_Type]})], ignore_index=True)
        else:
            global converted_unified_csv_frame_test
            converted_unified_csv_frame_test = pd.concat([converted_unified_csv_frame_test, pd.DataFrame({'Key': [t_Key], 'Type': [t_Type]})], ignore_index=True)
        

In [ ]:
for i in range (len(folders)):
    if 'lp' in folders[i].name:
        csv_dealer(folders[i],  'LP')
    elif 'vt' in folders[i].name:
        csv_dealer(folders[i], 'VT')
    else:
        csv_dealer(folders[i], 'Noise')



In [ ]:
# save converted_unified_csv_frame to csv file
converted_unified_csv_frame_test.to_csv(LP_VT_test_csv_file_path, index=False)
converted_unified_csv_frame_train.to_csv(LP_VT_train_csv_file_path, index=False)
# close LSD_hdf5
LPVT_hdf5.close()

# Step 2: Data preprocess

In [11]:
class VolcanicSeismicProcessor:
    """
    data dealer
    """
    def __init__(self, sampling_rate=50):
        self.sampling_rate = sampling_rate          # target rate of resampleing
        self.pca = None
        self.scaler = StandardScaler()

    def preprocess(self, data, original_sr=None, sensor_sensitivity=1.0):
        """
        preprocess step: resampling, centering, Offset correction
        data: shape: (n_samples, n_channels)
        original_sr: original sampling rate(Hz), None represent for no resampling
        """
        if original_sr is None:
            original_sr = self.sampling_rate

        if original_sr != self.sampling_rate:
            n_orig = data.shape[0]
            n_new = int(n_orig * self.sampling_rate / original_sr)
            resampled_channels = []
            for ch in range(data.shape[1]):
                # resample
                ch_resampled = signal.resample(data[:, ch], n_new, axis=0)
                resampled_channels.append(ch_resampled)
            data_resampled = np.column_stack(resampled_channels)
        else:
            data_resampled = data

        # 2. centering
        data_demeaned = data_resampled - np.mean(data_resampled, axis=0)

        # 3. offset correction
        data_corrected = data_demeaned / sensor_sensitivity
        return data_corrected

    def emd_decompose(self, signal_data, max_imf=10):
        """EMD method"""
        def find_extrema(x):
            max_idx = np.where((x[1:-1] > x[:-2]) & (x[1:-1] > x[2:]))[0] + 1
            min_idx = np.where((x[1:-1] < x[:-2]) & (x[1:-1] < x[2:]))[0] + 1
            return max_idx, min_idx

        def cubic_spline_interpolation(x, idx):
            if len(idx) < 2:
                return np.zeros_like(x)
            cs = CubicSpline(idx, x[idx], bc_type='natural')
            return cs(np.arange(len(x)))

        imfs = []
        residual = signal_data.copy()
        for _ in range(max_imf):
            h = residual.copy()
            sd_threshold = 0.3
            max_iter = 1000
            for _ in range(max_iter):
                max_idx, min_idx = find_extrema(h)
                if len(max_idx) < 2 or len(min_idx) < 2:
                    break
                upper = cubic_spline_interpolation(h, max_idx)
                lower = cubic_spline_interpolation(h, min_idx)
                mean_env = (upper + lower) / 2
                h_new = h - mean_env
                sd = np.sum((h - h_new)**2) / (np.sum(h_new**2) + 1e-10)
                h = h_new
                if sd < sd_threshold:
                    break
            imfs.append(h)
            residual = residual - h
            if len(find_extrema(residual)[0]) < 2:
                break
        return np.array(imfs)

    def select_imfs(self, imfs):
        """select top 3 the most important imfs"""
        if len(imfs) == 0:
            return None
        total_var = np.sum(np.var(imfs, axis=1))
        var_ratios = np.var(imfs, axis=1) / total_var
        top3_idx = np.argsort(var_ratios)[-3:][::-1]
        return imfs[top3_idx]

    def extract_features(self, imf):
        """
        get 54 features from each imfs
        """
        features = []

        # ---- Time domain features ----
        features.append(np.max(imf))
        features.append(np.min(imf))
        features.append(np.mean(imf))
        features.append(np.std(imf))
        features.append(np.sum(imf**2))                     # energy
        zero_crossings = np.sum(np.diff(np.signbit(imf)))
        features.append(zero_crossings / len(imf))          
        features.append(((imf - np.mean(imf))**4).mean() / (np.std(imf)**4 + 1e-10))  
        features.append(((imf - np.mean(imf))**3).mean() / (np.std(imf)**3 + 1e-10))  
        features.append(np.argmax(imf) / len(imf))         

        analytic = hilbert(imf)
        envelope = np.abs(analytic)
        features.append(np.mean(envelope))
        features.append(np.std(envelope))
        features.append(np.max(envelope))

        features.append(len(imf) / self.sampling_rate)     

        # ---- Frequency domain features（Welch PSD） ----
        freqs, psd = signal.welch(imf, fs=self.sampling_rate, nperseg=min(256, len(imf)//4))
        centroid = np.sum(freqs * psd) / (np.sum(psd) + 1e-10)
        features.append(centroid)
        bandwidth = np.sqrt(np.sum(((freqs - centroid)**2) * psd) / (np.sum(psd) + 1e-10))
        features.append(bandwidth)
        features.append(np.max(psd))                         
        features.append(freqs[np.argmax(psd)])               
        features.append(np.sum(psd))                          
        psd_norm = psd / (np.sum(psd) + 1e-10)
        spectral_entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-10))
        features.append(spectral_entropy)

        # ---- Mel-Frequency Cepstral Coefficients ----
        cepstrum = np.real(ifft(np.log(np.abs(fft(imf)) + 1e-10)))
        mfcc = cepstrum[:13]
        if len(mfcc) < 13:
            mfcc = np.pad(mfcc, (0, 13 - len(mfcc)))
        features.extend(mfcc[:13])

        # Ensure 54 features
        if len(features) < 54:
            features.extend([np.percentile(imf, 25), np.percentile(imf, 50), np.percentile(imf, 75),
                             np.mean(np.diff(imf)), np.std(np.diff(imf))])
        if len(features) > 54:
            features = features[:54]
        return np.array(features)

    def process_event(self, multichannel_data, original_sr=None, sensor_sensitivity=1.0):
        """
        Processing of each event
        Multichannel_data: (time, 3)
        """
        # 1. preprocess
        processed = self.preprocess(multichannel_data, original_sr, sensor_sensitivity)

        # 2. feature extraction
        all_features = []
        for ch in range(processed.shape[1]):          # 3 channels
            sig = processed[:, ch]
            imfs = self.emd_decompose(sig)
            if len(imfs) == 0:
                continue
            top_imfs = self.select_imfs(imfs)
            for imf in top_imfs:
                feats = self.extract_features(imf)    # 54 dimensions
                all_features.extend(feats)

        feature_vector = np.array(all_features)
        # suposed length: 3 channel × 3IMF × 54 = 486
        if len(feature_vector) != 486:
            if len(feature_vector) < 486:
                feature_vector = np.pad(feature_vector, (0, 486 - len(feature_vector)))
            else:
                feature_vector = feature_vector[:486]
        return feature_vector

    def prepare_features_for_models(self, events_list, labels_list, original_sr=None, sensor_sensitivities=None):
        """
        Feature matrix(nomalized + PCA)
        """
        features_matrix = []
        valid_labels = []

        for i, event in enumerate(events_list):
            if i % 50 == 0 and i > 0:
                print(f"{i}/{len(events_list)} events have been done")
            sensitivity = sensor_sensitivities[i] if sensor_sensitivities else 1.0
            feats = self.process_event(event, original_sr=original_sr, sensor_sensitivity=sensitivity)
            features_matrix.append(feats)
            valid_labels.append(labels_list[i])

        features_matrix = np.array(features_matrix)

        print("normalized feature...")
        features_scaled = self.scaler.fit_transform(features_matrix)

        # PCA dimension reduction
        print("PCA down to 200 dimension...")
        if self.pca is None:
            self.pca = PCA(n_components=200)
            features_pca = self.pca.fit_transform(features_scaled)
            print(f"PCA explained variance ratio: {np.sum(self.pca.explained_variance_ratio_):.3f}")
        else:
            features_pca = self.pca.transform(features_scaled)

        return features_pca, np.array(valid_labels)

In [8]:
# -------------------- load data --------------------
def load_data(h5_path, csv_path):
    # 1. read from CSV
    df = pd.read_csv(csv_path)
    ids = df['Key'].astype(str).values  
    types = df['Type'].values

    # 2. class mapping
    class_names = sorted(set(types))
    type_to_label = {name: i for i, name in enumerate(class_names)}

    events = []
    labels = []
    missing = []

    # 3. read from hdf5
    with h5py.File(h5_path, 'r') as f:
        for idx, typ in zip(ids, types):
            dset_name = idx
            if dset_name not in f:
                missing.append(dset_name)
                continue
            waveform = f[dset_name][:]          
            events.append(waveform)
            labels.append(type_to_label[typ])

    if missing:
        print(f"Dataset is missing: {missing}")

    return events, labels, class_names

# Step 3: Main function, include training and testing.

In [14]:
# -------------------- main --------------------
def main():
    # data path
    h5_file = "D:/seisbench_data/total.hdf5"          
    train_csv = "D:/seisbench_data/training.csv"
    test_csv = "D:/seisbench_data/test.csv"

    # load data
    print("Loading training set...")
    train_events, train_labels, class_names = load_data(h5_file, train_csv)
    print(f"Training set size: {len(train_events)}")
    print("Loading testing set...")
    test_events, test_labels, _ = load_data(h5_file, test_csv)
    print(f"Testing set size: {len(test_events)}")
    print("class:", class_names)

    # resampling to 50 Hz
    processor = VolcanicSeismicProcessor(sampling_rate=50)

    print("\nTraining set preparing...")
    X_train, y_train = processor.prepare_features_for_models(train_events, train_labels, original_sr=100)
    print("Testing set preparing...")
    X_test, y_test = processor.prepare_features_for_models(test_events, test_labels, original_sr=100)

    # SVM Training
    print("\nTraining SVM...")
    svm = SVC(kernel='rbf', C=10, gamma=0.002, probability=True, random_state=42)
    svm.fit(X_train, y_train)
    svm_pred = svm.predict(X_test)
    svm_acc = accuracy_score(y_test, svm_pred)
    print(f"SVM accuracy: {svm_acc*100:.2f}%")
    print("\nSVM classification report:")
    print(classification_report(y_test, svm_pred, target_names=class_names))

    # RF training（750 trees）
    print("\nTraining RF...")
    rf = RandomForestClassifier(n_estimators=750, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)
    rf_acc = accuracy_score(y_test, rf_pred)
    print(f"RF accuracy: {rf_acc*100:.2f}%")
    print("\nRF classification report:")
    print(classification_report(y_test, rf_pred, target_names=class_names))

    print("\nDone! ")


In [15]:
if __name__ == "__main__":
    main()

Loading training set...


Training set size: 57721
Loading testing set...
Testing set size: 24965
class: ['LP', 'Noise', 'VT']

Training set preparing...
50/57721 events have been done
100/57721 events have been done
150/57721 events have been done
200/57721 events have been done
250/57721 events have been done
300/57721 events have been done
350/57721 events have been done
400/57721 events have been done
450/57721 events have been done
500/57721 events have been done
550/57721 events have been done
600/57721 events have been done
650/57721 events have been done
700/57721 events have been done
750/57721 events have been done
800/57721 events have been done
850/57721 events have been done
900/57721 events have been done
950/57721 events have been done
1000/57721 events have been done
1050/57721 events have been done
1100/57721 events have been done
1150/57721 events have been done
1200/57721 events have been done
1250/57721 events have been done
1300/57721 events have been done
1350/57721 events have been done
1